In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp
from math import e
from scipy.optimize import minimize
from IPython.display import clear_output
from scipy.stats import norm
from scipy import optimize
import pickle
from scipy import integrate

In [2]:
best_model = 'model_Yates_Satter_1999'

In [3]:
with open('reconciled_data.pkl', 'rb') as f:
    data = pickle.load(f)

In [4]:
data['F_HCs_exp'] = data['F_C5+_exp'] + data['F_C1_C3_exp']

In [5]:
with open('alpha_params.pkl', 'rb') as f:
    alpha_params = pickle.load(f)

In [6]:
with open('K_Cn_params.pkl', 'rb') as f:
    K_Cn_params = pickle.load(f)
K_Cn_params

,slope,intercept
2,[[-0.0683038027220198]],[35.68897799525677]
3,[-0.1571538024026902],77.945402


K_Cn_params['slope'] = 0
K_Cn_params['intercept'] = 0
K_Cn_params

data[['F_C1_e', 'F_C2_e', 'F_C3_e', 'F_C7_e', 'F_C8_e', 'F_C9_e', 'F_C10_e', 'F_C11_e', 
      'F_C12_e', 'F_C13_e', 'F_C14_e', 'F_C15_e', 'F_C16_e', 'F_C17_e', 'F_C18_e',
      'F_C19_e', 'F_C20_e', 'F_C21_e', 'F_C22_e', 'F_C23_e', 'F_C24_e', 'F_C25_e', 
      'F_C26_e', 'F_C27_e', 'F_C28_e', 'F_C29_e', 'F_C30_e', 'F_C31_e', 'F_C32_e', 
      'F_C33_e', 'F_C34_e', 'F_C35_e', 'F_C36_e']] = 0

In [7]:
kinetic_data = data.loc[data['CINÉTICA'] == 1.0].reset_index()

In [8]:
from sklearn.model_selection import train_test_split

fitting_data, test_data = train_test_split(kinetic_data, test_size=0.4, random_state=105)
fitting_data = fitting_data.reset_index()
test_data = test_data.reset_index()

In [9]:
from models.model_power_law                             import model_power_law
from models.model_Yates_Satter_1999                     import model_Yates_Satter_1999
from models.model_Botes_2009                            import model_Botes_2009
from models.model_vanSteen_Schulz_1999                  import model_vanSteen_Schulz_1999
from models.model_Ojeda_2010                            import model_Ojeda_2010
from models.model_Mousavi_2015                          import model_Mousavi_2015
from models.model_Mousavi_2015_power_law                import model_Mousavi_2015_power_law
from models.model_Wang_2025_full                        import model_Wang_2025_full
from models.model_Wang_2025_simple                      import model_Wang_2025_simple

def models(model_n, data, params, passos, alpha_params, K_Cn_params):
    match model_n:
        case 'model_power_law':
            return model_power_law(data, params, passos, alpha_params, K_Cn_params)
        case 'model_Yates_Satter_1999':
            return model_Yates_Satter_1999(data, params, passos, alpha_params, K_Cn_params)
        case 'model_Botes_2009':
            return model_Botes_2009(data, params, passos, alpha_params, K_Cn_params)
        case 'model_vanSteen_Schulz_1999':
            return model_vanSteen_Schulz_1999(data, params, passos, alpha_params, K_Cn_params)
        case 'model_Ojeda_2010':
            return model_Ojeda_2010(data, params, passos, alpha_params, K_Cn_params)
        case 'model_Mousavi_2015':
            return model_Mousavi_2015(data, params, passos, alpha_params, K_Cn_params)
        case 'model_Mousavi_2015_power_law':
            return model_Mousavi_2015_power_law(data, params, passos, alpha_params, K_Cn_params)
        case 'model_Wang_2025_full':
            return model_Wang_2025_full(data, params, passos, alpha_params, K_Cn_params)
        case 'model_Wang_2025_simple':
            return model_Wang_2025_simple(data, params, passos, alpha_params, K_Cn_params)

In [10]:
with open('minimums.pkl', 'rb') as f:
    minimums = pickle.load(f)

In [11]:
# replace the best model parameters with the means from bootstrapping
with open('bootstrap_Mousavi_power_law/ci_list.pkl', 'rb') as f:
    ci_list_Mousavi_power_law = pickle.load(f)

minimums.at[minimums.index[minimums['model'] == 'model_Mousavi_2015_power_law'].values[0],
            'params'] = ci_list_Mousavi_power_law.loc['Mean'].values

FileNotFoundError: [Errno 2] No such file or directory: 'bootstrap_Mousavi_power_law/ci_list.pkl'

In [ ]:
# replace the best model parameters with the means from bootstrapping
with open('bootstrap_model_Yates_Satter_1999/ci_list.pkl', 'rb') as f:
    ci_list_Yates_Satter_1999 = pickle.load(f)

minimums.at[minimums.index[minimums['model'] == best_model].values[0],
            'params'] = ci_list_Yates_Satter_1999.loc['Mean'].values

In [ ]:
minimums

In [ ]:
p = minimums['params'].values
p

In [ ]:
A1 = []
E1 = []
for i in range(len(p)):
    A1.append(p[i][0])
    E1.append(p[i][1])
minimums['A1'] = A1
minimums['E1'] = E1

In [ ]:
A2 = [0, 
        0,
        0,
        0,
        minimums['params'][4][2],
        0,
        0,
        0,
        0,
       ]
minimums['A2'] = A2

In [ ]:
E2 = [0, 
        0,
        0,
        0,
        minimums['params'][4][3],
        0,
        0,
        0,
        0,
       ]
minimums['E2'] = E2

In [ ]:
k_CO = [0, 
        minimums['params'][1][2],
        minimums['params'][2][2],
        minimums['params'][3][2],
        minimums['params'][4][4],
        minimums['params'][5][2],
        minimums['params'][6][2],
        minimums['params'][7][2],
        minimums['params'][8][2],
       ]
minimums['k_CO'] = k_CO

In [ ]:
H_CO = [0, 
        minimums['params'][1][3],
        minimums['params'][2][3],
        minimums['params'][3][3],
        minimums['params'][4][5],
        minimums['params'][5][3],
        minimums['params'][6][3],
        minimums['params'][7][3],
        minimums['params'][8][3],
       ]
minimums['H_CO'] = H_CO

In [ ]:
k_H2O = [0, 
        0,
        0,
        0,
        0,
        0,
        0,
        minimums['params'][7][4],
        minimums['params'][8][4],
       ]
minimums['k_H2O'] = k_H2O

In [ ]:
H_H2O = [0, 
        0,
        0,
        0,
        0,
        0,
        0,
        minimums['params'][7][5],
        minimums['params'][8][5],
       ]
minimums['H_H2O'] = H_H2O

In [ ]:
a = [minimums['params'][0][2], 
        0,
        0,
        0,
        0,
        0,
        minimums['params'][6][4],
        minimums['params'][7][6],
        minimums['params'][8][6],
       ]
minimums['a'] = a

In [ ]:
b = [minimums['params'][0][3], 
        0,
        0,
        0,
        0,
        0,
        minimums['params'][6][5],
        minimums['params'][7][7],
        minimums['params'][8][7],
       ]
minimums['b'] = b

In [ ]:
c = [0, 
        0,
        0,
        0,
        0,
        0,
        minimums['params'][6][6],
        minimums['params'][7][8],
        0,
       ]
minimums['c'] = c

In [ ]:
d = [0, 
        0,
        0,
        0,
        0,
        0,
        0,
        minimums['params'][7][9],
        0,
       ]
minimums['d'] = d

In [ ]:

minimums_to_LaTeX = minimums[['model', 'aic_test', 
                             'rel_lik_test', 'A1', 'E1', 'A2', 'E2', 'k_CO', 'H_CO',
                                      'k_H2O', 'H_H2O', 'a', 'b', 'c', 'd']]
minimums_to_LaTeX['n'] = minimums.index.values + 1
minimums_to_LaTeX = minimums_to_LaTeX[['n',
                             'aic_test', 
                             'rel_lik_test', 'A1', 'E1', 'A2', 'E2', 'k_CO', 'H_CO',
                                      'k_H2O', 'H_H2O', 'a', 'b', 'c', 'd'
                                      ]]

minimums_to_LaTeX['aic_test'] = minimums_to_LaTeX['aic_test'].map('{:.0f}'.format)
minimums_to_LaTeX['rel_lik_test'] = minimums_to_LaTeX['rel_lik_test'].map('{:.0e}'.format)
minimums_to_LaTeX['A1'] = minimums_to_LaTeX['A1'].map('{:.2e}'.format)
minimums_to_LaTeX['E1'] = minimums_to_LaTeX['E1'].map('{:.2e}'.format)
minimums_to_LaTeX['A2'] = minimums_to_LaTeX['A2'].map('{:.2e}'.format)
minimums_to_LaTeX['E2'] = minimums_to_LaTeX['E2'].map('{:.2e}'.format)
minimums_to_LaTeX['k_CO'] = minimums_to_LaTeX['k_CO'].map('{:.2e}'.format)
minimums_to_LaTeX['H_CO'] = minimums_to_LaTeX['H_CO'].map('{:.2e}'.format)
minimums_to_LaTeX['k_H2O'] = minimums_to_LaTeX['k_H2O'].map('{:.2e}'.format)
minimums_to_LaTeX['H_H2O'] = minimums_to_LaTeX['H_H2O'].map('{:.2e}'.format)
minimums_to_LaTeX['a'] = minimums_to_LaTeX['a'].map('{:.1f}'.format)
minimums_to_LaTeX['b'] = minimums_to_LaTeX['b'].map('{:.1f}'.format)
minimums_to_LaTeX['c'] = minimums_to_LaTeX['c'].map('{:.1f}'.format)
minimums_to_LaTeX['d'] = minimums_to_LaTeX['d'].map('{:.1f}'.format)

# Use the Styler to format numbers
minimums_to_LaTeX_style = {
    'aic_test': lambda x: f'\\num{{{x}}}',
    'rel_lik_test': lambda x: f'\\num{{{x}}}',
    'A1': lambda x: f'\\num{{{x}}}',
    'E1': lambda x: f'\\num{{{x}}}',
    'A2': lambda x: f'\\num{{{x}}}',
    'E2': lambda x: f'\\num{{{x}}}',
    'k_CO': lambda x: f'\\num{{{x}}}',
    'H_CO': lambda x: f'\\num{{{x}}}',
    'k_H2O': lambda x: f'\\num{{{x}}}',
    'H_H2O': lambda x: f'\\num{{{x}}}',
    'a': lambda x: f'\\num{{{x}}}',
    'b': lambda x: f'\\num{{{x}}}',
    'c': lambda x: f'\\num{{{x}}}',
    'd': lambda x: f'\\num{{{x}}}'
}


minimums_to_LaTeX = minimums_to_LaTeX[['n', 'A1', 'E1', 'A2', 'E2', 'k_CO', 'H_CO',
                                      'k_H2O', 'H_H2O', 'a', 'b', 'c', 'd']].to_latex(
#                             escape = True, 
                             index = False, 
                             float_format=None,
                             formatters = minimums_to_LaTeX_style
                                              )
print(minimums_to_LaTeX)

with open("/home/rafael/GoogleDrive/uem/Doutorado/Papers/SAF_kinetics/tables/AICs.tex", "w") as f:
    f.write(minimums_to_LaTeX)

In [ ]:
parameters = minimums.loc[minimums['model'] == best_model]['params'].values[-1]
parameters

In [ ]:
kinetic_data['alpha_model'] = 1/(1 + e**-(alpha_params[0] + alpha_params[1]*kinetic_data['T_R_K']
                                       + alpha_params[2]*kinetic_data['F_H2_e_mol_s']/kinetic_data['F_CO_e_mol_s']))

In [12]:
test = models(best_model, test_data, parameters, passos:=200, alpha_params, K_Cn_params)

NameError: name 'parameters' is not defined

In [ ]:
test['X_CO_model'] = (test_data['F_CO_e_mol_s'] - 
                            test['F_CO_s_model'])/test_data['F_CO_e_mol_s']
test['X_H2_model'] = (test_data['F_H2_e_mol_s'] - 
                            test['F_H2_s_model'])/test_data['F_H2_e_mol_s']

In [ ]:
test_ASF_F = pd.DataFrame()
test_ASF_F['F_C1'] = test['F_C1_model']
test_ASF_F['F_C2'] = test['F_C2_model'] + test['F_Olef_C2_model']
test_ASF_F['F_C3'] = test['F_C3_model']
test_ASF_F['F_C7'] = test['F_C7_model'] + test['F_Olef_C7_model']
test_ASF_F['F_C8'] = test['F_C8_model'] + test['F_Olef_C8_model']
test_ASF_F['F_C9'] = test['F_C9_model'] + test['F_Olef_C9_model']
test_ASF_F['F_C10'] = test['F_C10_model'] + test['F_Olef_C10_model']
test_ASF_F['F_C11'] = test['F_C11_model'] + test['F_Olef_C11_model']
test_ASF_F['F_C12'] = test['F_C12_model'] + test['F_Olef_C12_model']
test_ASF_F['F_C13'] = test['F_C13_model'] + test['F_Olef_C13_model']
test_ASF_F['F_C14'] = test['F_C14_model'] + test['F_Olef_C14_model']
test_ASF_F['F_C15'] = test['F_C15_model'] + test['F_Olef_C15_model']
test_ASF_F['F_C16'] = test['F_C16_model'] + test['F_Olef_C16_model']
test_ASF_F['F_C17'] = test['F_C17_model'] + test['F_Olef_C17_model']
test_ASF_F['F_C18'] = test['F_C18_model'] + test['F_Olef_C18_model']
test_ASF_F['F_C19'] = test['F_C19_model']
test_ASF_F['F_C20'] = test['F_C20_model']
test_ASF_F['F_C21'] = test['F_C21_model']
test_ASF_F['F_C22'] = test['F_C22_model']
test_ASF_F['F_C23'] = test['F_C23_model']
test_ASF_F['F_C24'] = test['F_C24_model']
test_ASF_F['F_C25'] = test['F_C25_model']
test_ASF_F['F_C26'] = test['F_C26_model']
test_ASF_F['F_C27'] = test['F_C27_model']
test_ASF_F['F_C28'] = test['F_C28_model']
test_ASF_F['F_C29'] = test['F_C29_model']
test_ASF_F['F_C30'] = test['F_C30_model']
test_ASF_F['F_C31'] = test['F_C31_model']
test_ASF_F['F_C32'] = test['F_C32_model']
test_ASF_F['F_C33'] = test['F_C33_model']
test_ASF_F['F_C34'] = test['F_C34_model']
test_ASF_F['F_C35'] = test['F_C35_model']

In [ ]:
test['T_R_C'] = test['T_out'] - 273

In [ ]:
data_ASF_F = pd.DataFrame()
data_ASF_F['F_C1'] = test_data['F_CH4_s_mol_s']
data_ASF_F['F_C2'] = test_data['F_C2H6_s_mol_s'] + test_data['F_C2H4_s_mol_s']
data_ASF_F['F_C3'] = test_data['F_C3H8_s_mol_s']
data_ASF_F['F_C7'] = test_data['F_C7'] + test_data['F_Olef_C7']
data_ASF_F['F_C8'] = test_data['F_C8'] + test_data['F_Olef_C8']
data_ASF_F['F_C9'] = test_data['F_C9'] + test_data['F_Olef_C9']
data_ASF_F['F_C10'] = test_data['F_C10'] + test_data['F_Olef_C10']
data_ASF_F['F_C11'] = test_data['F_C11'] + test_data['F_Olef_C11']
data_ASF_F['F_C12'] = test_data['F_C12'] + test_data['F_Olef_C12']
data_ASF_F['F_C13'] = test_data['F_C13'] + test_data['F_Olef_C13']
data_ASF_F['F_C14'] = test_data['F_C14'] + test_data['F_Olef_C14']
data_ASF_F['F_C15'] = test_data['F_C15'] + test_data['F_Olef_C15']
data_ASF_F['F_C16'] = test_data['F_C16'] + test_data['F_Olef_C16']
data_ASF_F['F_C17'] = test_data['F_C17'] + test_data['F_Olef_C17']
data_ASF_F['F_C18'] = test_data['F_C18'] + test_data['F_Olef_C18']
data_ASF_F['F_C19'] = test_data['F_C19']
data_ASF_F['F_C20'] = test_data['F_C20']
data_ASF_F['F_C21'] = test_data['F_C21']
data_ASF_F['F_C22'] = test_data['F_C22']
data_ASF_F['F_C23'] = test_data['F_C23']
data_ASF_F['F_C24'] = test_data['F_C24']
data_ASF_F['F_C25'] = test_data['F_C25']
data_ASF_F['F_C26'] = test_data['F_C26']
data_ASF_F['F_C27'] = test_data['F_C27']
data_ASF_F['F_C28'] = test_data['F_C28']
data_ASF_F['F_C29'] = test_data['F_C29']
data_ASF_F['F_C30'] = test_data['F_C30']
data_ASF_F['F_C31'] = test_data['F_C31']
data_ASF_F['F_C32'] = test_data['F_C32']
data_ASF_F['F_C33'] = test_data['F_C33']
data_ASF_F['F_C34'] = test_data['F_C34']
data_ASF_F['F_C35'] = test_data['F_C35']


data_ASF_y = data_ASF_F.mean()/data_ASF_F.mean().sum()

test_ASF_y = test_ASF_F.mean()/test_ASF_F.mean().sum()

test_ASF = pd.DataFrame()
test_ASF['y_model_mean'] = test_ASF_y.values
test_ASF['y_data_mean'] = data_ASF_y.values
test_ASF['y_data_mean'] = data_ASF_y.values


test_ASF['n_C'] = np.concat(([1,2,3],list(range(7,36))))

In [ ]:
data_ASF_y_dist = data_ASF_F.dropna().div(data_ASF_F.dropna().sum(axis=1), axis=0)

In [ ]:
test_ASF_F_210_C = test_ASF_F.loc[(  (test['T_R_C'] > 210) 
                                    & (test['T_R_C'] < 220))]
data_ASF_F_210_C = data_ASF_F.loc[(  (test['T_R_C'] > 210) 
                                    & (test['T_R_C'] < 220))]
test_ASF_y_210_C = test_ASF_F_210_C.mean()/test_ASF_F_210_C.mean().sum()
data_ASF_y_210_C = data_ASF_F_210_C.mean()/data_ASF_F_210_C.mean().sum()
test_ASF_210_C = pd.DataFrame()
test_ASF_210_C['y_model_mean'] = test_ASF_y_210_C.values
test_ASF_210_C['y_data_mean'] = data_ASF_y_210_C.values
test_ASF_210_C['y_data_mean'] = data_ASF_y_210_C.values
test_ASF_210_C['n_C'] = np.concat(([1,2,3],list(range(7,36))))
test_ASF_210_C['T_R_C'] = 210
data_ASF_y_dist_210_C = data_ASF_F_210_C.dropna().div(data_ASF_F_210_C.dropna().sum(axis=1), axis=0)

In [ ]:
test_ASF_F_220_C = test_ASF_F.loc[(  (test['T_R_C'] > 220) 
                                    & (test['T_R_C'] < 230))]
data_ASF_F_220_C = data_ASF_F.loc[(  (test['T_R_C'] > 220) 
                                    & (test['T_R_C'] < 230))]
test_ASF_y_220_C = test_ASF_F_220_C.mean()/test_ASF_F_220_C.mean().sum()
data_ASF_y_220_C = data_ASF_F_220_C.mean()/data_ASF_F_220_C.mean().sum()
test_ASF_220_C = pd.DataFrame()
test_ASF_220_C['y_model_mean'] = test_ASF_y_220_C.values
test_ASF_220_C['y_data_mean'] = data_ASF_y_220_C.values
test_ASF_220_C['y_data_mean'] = data_ASF_y_220_C.values
test_ASF_220_C['n_C'] = np.concat(([1,2,3],list(range(7,36))))
test_ASF_220_C['T_R_C'] = 220
data_ASF_y_dist_220_C = data_ASF_F_220_C.dropna().div(data_ASF_F_220_C.dropna().sum(axis=1), axis=0)

In [ ]:
test_ASF_F_230_C = test_ASF_F.loc[(  (test['T_R_C'] > 230) 
                                    & (test['T_R_C'] < 240))]
data_ASF_F_230_C = data_ASF_F.loc[(  (test['T_R_C'] > 230) 
                                    & (test['T_R_C'] < 240))]
test_ASF_y_230_C = test_ASF_F_230_C.mean()/test_ASF_F_230_C.mean().sum()
data_ASF_y_230_C = data_ASF_F_230_C.mean()/data_ASF_F_230_C.mean().sum()
test_ASF_230_C = pd.DataFrame()
test_ASF_230_C['y_model_mean'] = test_ASF_y_230_C.values
test_ASF_230_C['y_data_mean'] = data_ASF_y_230_C.values
test_ASF_230_C['y_data_mean'] = data_ASF_y_230_C.values
test_ASF_230_C['n_C'] = np.concat(([1,2,3],list(range(7,36))))
test_ASF_230_C['T_R_C'] = 230
data_ASF_y_dist_230_C = data_ASF_F_230_C.dropna().div(data_ASF_F_230_C.dropna().sum(axis=1), axis=0)

In [ ]:
query = ((test['T_R_C'] > 240) 
         & (test['T_R_C'] < 250))
test_ASF_F_240_C = test_ASF_F.loc[query]
data_ASF_F_240_C = data_ASF_F.loc[query]
test_ASF_y_240_C = test_ASF_F_240_C.mean()/test_ASF_F_240_C.mean().sum()
data_ASF_y_240_C = data_ASF_F_240_C.mean()/data_ASF_F_240_C.mean().sum()
test_ASF_240_C = pd.DataFrame()
test_ASF_240_C['y_model_mean'] = test_ASF_y_240_C.values
test_ASF_240_C['y_data_mean'] = data_ASF_y_240_C.values
test_ASF_240_C['y_data_mean'] = data_ASF_y_240_C.values
test_ASF_240_C['n_C'] = np.concat(([1,2,3],list(range(7,36))))
test_ASF_240_C['T_R_C'] = 240
data_ASF_y_dist_240_C = data_ASF_F_240_C.dropna().div(data_ASF_F_240_C.dropna().sum(axis=1), axis=0)

In [ ]:
test_ASF = pd.concat((test_ASF_210_C,test_ASF_220_C,test_ASF_230_C,test_ASF_240_C))

In [ ]:
test_ASF.to_pickle('test_ASF_distribution.pkl')

In [ ]:
fig, ax = plt.subplots(2,2,figsize=(8,6))
plt.subplots_adjust(hspace=0.4, wspace = 0.3)

# 210 ºC
sns.lineplot(
                data=test_ASF_210_C, 
                x="n_C", 
                y="y_model_mean", 
                ax = ax[0,0],
#                 label = 'Mean ASF model prediction',
                color = 'blue'
)
# Calculate st deviation
std = data_ASF_y_dist_210_C.std()
# Add error bars
ax[0,0].errorbar(test_ASF_210_C['n_C'], test_ASF_210_C['y_data_mean'], 
             yerr=std, fmt='o', color='blue', label='Mean exp. conc. ± SD',
            lw = 0.8, markersize=3)
ax[0,0].set_title('210 ºC')

# 220 ºC
sns.lineplot(
                data=test_ASF_220_C, 
                x="n_C", 
                y="y_model_mean", 
                ax = ax[0,1],
#                 label = 'Mean ASF model prediction',
                color = 'blue'
)
# Calculate st deviation
std = data_ASF_y_dist_220_C.std()
# Add error bars
ax[0,1].errorbar(test_ASF_220_C['n_C'], test_ASF_220_C['y_data_mean'], 
             yerr=std, fmt='o', color='blue', label='Mean exp. conc. ± SD',
            lw = 0.8, markersize=3)
ax[0,1].set_title('220 ºC')

# 230 ºC
sns.lineplot(
                data=test_ASF_230_C, 
                x="n_C", 
                y="y_model_mean", 
                ax = ax[1,0],
#                 label = 'Mean ASF model prediction',
                color = 'blue'
)
# Calculate st deviation
std = data_ASF_y_dist_230_C.std()
# Add error bars
ax[1,0].errorbar(test_ASF_230_C['n_C'], test_ASF_230_C['y_data_mean'], 
             yerr=std, fmt='o', color='blue', label='Mean exp. conc. ± SD',
            lw = 0.8, markersize=3)
ax[1,0].set_title('230 ºC')

# 240 ºC
sns.lineplot(
                data=test_ASF_240_C, 
                x="n_C", 
                y="y_model_mean", 
                ax = ax[1,1],
                label = 'Mean ASF model prediction',
                color = 'blue'
)
# Calculate st deviation
std = data_ASF_y_dist_240_C.std()
# Add error bars
ax[1,1].errorbar(test_ASF_240_C['n_C'], test_ASF_240_C['y_data_mean'], 
             yerr=std, fmt='o', color='blue', label='Mean exp. conc. ± SD',
            lw = 0.8, markersize=3)
ax[1,1].set_title('240 ºC')

for i in range(2):
    for j in range(2):
        ax[i,j].grid('both')
        ax[i,j].set_xlabel('chain length')
        ax[i,j].set_ylabel('molar fraction')
#         ax[i,j].set_yscale('log')
ax[1,1].legend(bbox_to_anchor=(0.3, -0.2))
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3,2.5))

# 210 ºC
sns.lineplot(
                data=test_ASF, 
                x="n_C", 
                y="y_model_mean", 
                ax = ax,
                label = 'ASF model prediction',
                color = 'blue'
)
sns.scatterplot(
                data=test_ASF, 
                x="n_C", 
                y="y_data_mean", 
                ax = ax,
                label = 'Experimental data',
                color = 'blue', s = 8
)
plt.grid('both')
plt.ylabel('$y_n$')
plt.xlabel('$n_C$')
# plt.yscale('log')
# plt.ylim(-0.05,1.05)
fig_name = 'ASF_plot.pdf'
plt.savefig('/home/rafael/GoogleDrive/uem/Doutorado/Papers/SAF_kinetics/figures/'+fig_name, bbox_inches = 'tight')

In [ ]:
fig, ax = plt.subplots(figsize=(3.5,3))
sns.scatterplot(
                data=kinetic_data, 
                x="alpha", 
                y="alpha_model",
                size = 'T_R_C',
#                 size = '$H_2/CO$',
                ax = ax,
)

plt.plot(np.linspace(0,1,
                     num = 10), 
         np.linspace(0,1,
                     num = 10))
plt.grid('both')
# plt.legend()
plt.legend(bbox_to_anchor=(1.02, 1), title = 'T (ºC)')
# plt.ticklabel_format(axis='both', style='sci', scilimits=(0,0))
plt.xlabel(r'$\alpha_{experimental}$')
plt.ylabel(r'$\alpha_{model}$')
# plt.yscale('log')
# plt.xscale('log')
# plt.xlim(0,1)
# plt.ylim(0,1)
fig_name = 'ASF_parity.pdf'
plt.savefig('/home/rafael/GoogleDrive/uem/Doutorado/Papers/SAF_kinetics/figures/'+fig_name, bbox_inches = 'tight')

In [ ]:
test['F_C1_C3_model'] = test['F_C1_model'] + test['F_C2_model'] + test['F_C3_model'] + test['F_Olef_C2_model']
test['F_C5+_model'] = test[[
     'F_Olef_C7_model', 'F_Olef_C8_model', 
     'F_Olef_C9_model', 'F_Olef_C10_model', 
     'F_Olef_C11_model', 'F_Olef_C12_model', 
     'F_Olef_C13_model', 'F_Olef_C14_model', 
     'F_Olef_C15_model', 'F_Olef_C16_model', 
     'F_Olef_C17_model', 'F_Olef_C18_model',
        
    
       'F_C7_model',
       'F_C8_model', 'F_C9_model', 'F_C10_model', 'F_C11_model', 'F_C12_model',
       'F_C13_model', 'F_C14_model', 'F_C15_model', 'F_C16_model',
       'F_C17_model', 'F_C18_model', 'F_C19_model', 'F_C20_model',
       'F_C21_model', 'F_C22_model', 'F_C23_model', 'F_C24_model',
       'F_C25_model', 'F_C26_model', 'F_C27_model', 'F_C28_model',
       'F_C29_model', 'F_C30_model', 'F_C31_model', 'F_C32_model',
       'F_C33_model', 'F_C34_model', 'F_C35_model', 'F_C36_model']].sum(axis=1)

test_plot_F = test[['F_CO_s_model', 'F_H2_s_model','F_H2O_s_model', 'F_CO2_s_model', 
                                'F_C1_C3_model', 'F_C5+_model']].melt()
test_plot_F = test_plot_F.replace('F_CO_s_model', '$CO$')
test_plot_F = test_plot_F.replace('F_CO2_s_model', '$CO_2$')
test_plot_F = test_plot_F.replace('F_H2_s_model', '$H_2$')
test_plot_F = test_plot_F.replace('F_H2O_s_model', '$H_2O$')
test_plot_F = test_plot_F.replace('F_C1_C3_model', '$C_1~to~C_3$')
test_plot_F = test_plot_F.replace('F_C5+_model', '$C_5+$')
# test_plot_F = test_plot_F.replace('F_CO2_s_model', '$CO_2$')


test_plot_F.rename(columns={'variable': 'Component', 'value': '$F_{out,model}$ (mol/s)'}, inplace=True)
test_plot_F['$F_{out,exp}$ (mol/s)'] = test_data[['F_CO_s_mol_s', 'F_H2_s_mol_s',
                                                             'F_H2O_s_mol_s', 'F_CO2_s_mol_s', 
                                                              'F_C1_C3_exp', 'F_C5+_exp']].melt()['value']
test_plot_F['$T$ (ºC)'] = test_data[['F_CO_s_mol_s', 'F_H2_s_mol_s', 
                                                 'F_H2O_s_mol_s', 'F_CO2_s_mol_s', 'F_C1_C3_exp', 'F_C5+_exp',
                                             'T_R_C']].melt(id_vars = 'T_R_C',)['T_R_C']

In [ ]:
test_plot_X = test[['X_CO_model', 'X_H2_model']].melt()
test_plot_X = test_plot_X.replace('X_CO_model', '$CO$')
test_plot_X = test_plot_X.replace('X_H2_model', '$H_2$')

test_plot_X.rename(columns={'variable': 'Component', 'value': '$X_{model}$'}, inplace=True)
test_plot_X['$X_{exp}$'] = test_data[['X_CO', 'X_H2']].melt()['value']
test_plot_X['$T$ (ºC)'] = test_data[['X_CO', 'X_H2', 'T_R_C']].melt(id_vars = 'T_R_C',)['T_R_C']

In [ ]:
fig, ax = plt.subplots(figsize=(3.5,3))
sns.scatterplot(
                data=test_plot_F.loc[test_plot_F['Component']!='$CO_2$'], 
                x="$F_{out,exp}$ (mol/s)", 
                y="$F_{out,model}$ (mol/s)", 
                hue = 'Component', 
                size="$T$ (ºC)",
        #     sizes=(10, 200),
                palette = "bright",
                ax = ax,
                style='Component',
)
plt.plot(np.linspace(test_plot_F['$F_{out,model}$ (mol/s)'].min()*0.5,
                     test_plot_F['$F_{out,model}$ (mol/s)'].max()*1.1,
                     num = 10), 
         np.linspace(test_plot_F['$F_{out,model}$ (mol/s)'].min()*0.5,
                     test_plot_F['$F_{out,model}$ (mol/s)'].max()*1.1,
                     num = 10))
plt.grid('both')
plt.legend(bbox_to_anchor=(1.02, 1.05))
plt.ticklabel_format(axis='both', style='sci', scilimits=(0,0))
plt.yscale('log')
plt.xscale('log')
# plt.xlim(1e-8,2e-4)
# plt.ylim(1e-8,2e-4)
fig_name = 'test_parity_F_out.pdf'
plt.savefig('/home/rafael/GoogleDrive/uem/Doutorado/Papers/SAF_kinetics/figures/'+fig_name, bbox_inches = 'tight')

In [ ]:
test_plot_F['Residuals (mol/s)'] = test_plot_F['$F_{out,exp}$ (mol/s)'] - test_plot_F['$F_{out,model}$ (mol/s)']

In [ ]:
fig, ax = plt.subplots(figsize=(3.5,3))
plt.grid('both')
sns.histplot(
                data=test_plot_F, 
                x="Residuals (mol/s)", 
#                 bins = 5,
#                 kde = True,
                ax = ax,
)
fig_name = 'test_hist_res_F.pdf'
plt.savefig('/home/rafael/GoogleDrive/uem/Doutorado/Papers/SAF_kinetics/figures/'+fig_name, bbox_inches = 'tight')

In [ ]:
fig, ax = plt.subplots(figsize=(3.5,3))
sns.scatterplot(
                data=test_plot_X, 
                x="$X_{exp}$", 
                y="$X_{model}$", 
                hue = 'Component', size="$T$ (ºC)",
        #     sizes=(10, 200),
                palette = "bright",
                ax = ax,
                style='Component',
)
plt.plot(np.linspace(0,1,
                     num = 10), 
         np.linspace(0,1,
                     num = 10))
plt.grid('both')
# plt.yscale('log')
# plt.xscale('log')
plt.legend(bbox_to_anchor=(1.02, 1))

fig_name = 'test_parity_X.pdf'
plt.savefig('/home/rafael/GoogleDrive/uem/Doutorado/Papers/SAF_kinetics/figures/'+fig_name, bbox_inches = 'tight')

In [ ]:
kinetics_solution = models(best_model, fitting_data, parameters, passos:=200, alpha_params, K_Cn_params)

In [ ]:
kinetics_solution['F_C1_C3_model'] = (
                                    kinetics_solution['F_C1_model'] 
                                    + kinetics_solution['F_C2_model'] 
                                    + kinetics_solution['F_Olef_C2_model'] 
                                    + kinetics_solution['F_C3_model']
)
kinetics_solution['F_C5+_model'] = kinetics_solution[[
       'F_Olef_C7_model', 'F_Olef_C8_model', 
     'F_Olef_C9_model', 'F_Olef_C10_model', 
     'F_Olef_C11_model', 'F_Olef_C12_model', 
     'F_Olef_C13_model', 'F_Olef_C14_model', 
     'F_Olef_C15_model', 'F_Olef_C16_model', 
     'F_Olef_C17_model', 'F_Olef_C18_model',
    
        'F_C7_model',
       'F_C8_model', 'F_C9_model', 'F_C10_model', 'F_C11_model', 'F_C12_model',
       'F_C13_model', 'F_C14_model', 'F_C15_model', 'F_C16_model',
       'F_C17_model', 'F_C18_model', 'F_C19_model', 'F_C20_model',
       'F_C21_model', 'F_C22_model', 'F_C23_model', 'F_C24_model',
       'F_C25_model', 'F_C26_model', 'F_C27_model', 'F_C28_model',
       'F_C29_model', 'F_C30_model', 'F_C31_model', 'F_C32_model',
       'F_C33_model', 'F_C34_model', 'F_C35_model', 'F_C36_model']].sum(axis=1)

In [ ]:
kinetics_plot_F = kinetics_solution[['F_CO_s_model', 'F_H2_s_model', 'F_H2O_s_model', 
                                'F_C1_C3_model', 'F_C5+_model']].melt()
kinetics_plot_F = kinetics_plot_F.replace('F_CO_s_model', '$CO$')
kinetics_plot_F = kinetics_plot_F.replace('F_H2_s_model', '$H_2$')
kinetics_plot_F = kinetics_plot_F.replace('F_H2O_s_model', '$H_2O$')
kinetics_plot_F = kinetics_plot_F.replace('F_C1_C3_model', '$C_1~to~C_3$')
kinetics_plot_F = kinetics_plot_F.replace('F_C5+_model', '$C_5+$')
# kinetics_plot_F = kinetics_plot_F.replace('F_CO2_s_model', '$CO_2$')


kinetics_plot_F.rename(columns={'variable': 'Component', 'value': '$F_{out,model}$ (mol/s)'}, inplace=True)
kinetics_plot_F['$F_{out,exp}$ (mol/s)'] = fitting_data[['F_CO_s_mol_s', 'F_H2_s_mol_s',
                                                             'F_H2O_s_mol_s', 
                                                              'F_C1_C3_exp', 'F_C5+_exp']].melt()['value']
kinetics_plot_F['$T$ (ºC)'] = fitting_data[['F_CO_s_mol_s', 'F_H2_s_mol_s', 
                                                 'F_H2O_s_mol_s', 'F_C1_C3_exp', 'F_C5+_exp',
                                             'T_R_C']].melt(id_vars = 'T_R_C',)['T_R_C']

In [ ]:
fig, ax = plt.subplots(figsize=(3.5,3))
sns.scatterplot(
                data=kinetics_plot_F, 
                x="$F_{out,exp}$ (mol/s)", 
                y="$F_{out,model}$ (mol/s)", 
                hue = 'Component', size="$T$ (ºC)",
        #     sizes=(10, 200),
                palette = "bright",
                ax = ax,
                style='Component',
)
plt.plot(np.linspace(kinetics_plot_F['$F_{out,model}$ (mol/s)'].min()*0.5,
                     kinetics_plot_F['$F_{out,model}$ (mol/s)'].max()*1.2,
                     num = 10), 
         np.linspace(kinetics_plot_F['$F_{out,model}$ (mol/s)'].min()*0.5,
                     kinetics_plot_F['$F_{out,model}$ (mol/s)'].max()*1.2,
                     num = 10))
plt.grid('both')
plt.legend(bbox_to_anchor=(1.02, 1))
plt.ticklabel_format(axis='both', style='sci', scilimits=(0,0))
# plt.xlim(0,kinetics_plot_F['$F_{out,model}$ (mol/s)'].max()*1.2)
# plt.ylim(0,kinetics_plot_F['$F_{out,model}$ (mol/s)'].max()*1.2)
# plt.yscale('log')
# plt.xscale('log')
fig_name = 'kinetics_parity_F_out.pdf'
plt.savefig('/home/rafael/GoogleDrive/uem/Doutorado/Papers/SAF_kinetics/figures/'+fig_name, bbox_inches = 'tight')

In [ ]:
data_C5_sel = pd.read_excel('C5+_mass_frac.ods')

In [ ]:
fig, ax = plt.subplots(figsize=(3,2.5))
sns.scatterplot(x = 'n_C_soma', y = 'Dry 15 bar', marker = 'o', label = 'Dry 15 bar',
             data=data_C5_sel, ax = ax)
sns.scatterplot(x = 'n_C_soma', y = 'Dry 20 bar', marker = 's', label = 'Dry 20 bar',
             data=data_C5_sel, ax = ax)
sns.scatterplot(x = 'n_C_soma', y = 'Dry 22,5 bar', marker = 'D', label = 'Dry 22.5 bar',
             data=data_C5_sel, ax = ax)
sns.scatterplot(x = 'n_C_soma', y = 'Dry 25 bar', marker = '^', label = 'Dry 25 bar',
             data=data_C5_sel, ax = ax)
sns.scatterplot(x = 'n_C_soma', y = 'Toluene 20 bar', marker = 'P', label = 'Toluene 20 bar',
             data=data_C5_sel, ax = ax)

plt.grid('both')
plt.xlabel('chain length')
plt.ylabel('mass fraction (g/g)')
plt.legend(title = 'Hot trap condition', bbox_to_anchor=(1.02, 1))
fig_name = 'C5_select.pdf'
plt.savefig('/home/rafael/GoogleDrive/uem/Doutorado/Papers/SAF_kinetics/figures/'+fig_name, 
            bbox_inches = 'tight')
plt.show()

In [ ]:
minimums['model']

In [ ]:
minimums

In [ ]:
models_results = pd.DataFrame()
for model in minimums['model']:
    solution = models(model, 
                                test_data, 
                                minimums['params'].loc[minimums['model'] == model].values[0], 
                                passos:=200, 
                                alpha_params, 
                                K_Cn_params)
    solution['model'] = model
    solution['F_H2_s_exp'] = test_data['F_H2_s_mol_s']
    solution['F_CO_s_exp'] = test_data['F_CO_s_mol_s']
    solution['F_H2O_s_exp'] = test_data['F_H2O_s_mol_s']
    solution['F_C1_C3_exp'] = test_data['F_C1_C3_exp']
    solution['F_C5+_exp'] = test_data['F_C5+_exp']
    solution['F_HCs_exp'] = test_data['F_C5+_exp'] + test_data['F_C1_C3_exp']
    models_results = pd.concat((models_results, solution))

In [ ]:
models_results

In [ ]:
models_results['F_C1_C3_model'] = (models_results['F_C1_model'] + models_results['F_C2_model'] 
                                   + models_results['F_C3_model'] + models_results['F_Olef_C2_model'])
models_results['F_C5+_model'] = models_results[[
     'F_Olef_C7_model', 'F_Olef_C8_model', 
     'F_Olef_C9_model', 'F_Olef_C10_model', 
     'F_Olef_C11_model', 'F_Olef_C12_model', 
     'F_Olef_C13_model', 'F_Olef_C14_model', 
     'F_Olef_C15_model', 'F_Olef_C16_model', 
     'F_Olef_C17_model', 'F_Olef_C18_model',
        
    
       'F_C7_model',
       'F_C8_model', 'F_C9_model', 'F_C10_model', 'F_C11_model', 'F_C12_model',
       'F_C13_model', 'F_C14_model', 'F_C15_model', 'F_C16_model',
       'F_C17_model', 'F_C18_model', 'F_C19_model', 'F_C20_model',
       'F_C21_model', 'F_C22_model', 'F_C23_model', 'F_C24_model',
       'F_C25_model', 'F_C26_model', 'F_C27_model', 'F_C28_model',
       'F_C29_model', 'F_C30_model', 'F_C31_model', 'F_C32_model',
       'F_C33_model', 'F_C34_model', 'F_C35_model', 'F_C36_model']].sum(axis=1)
models_results['F_HCs_model'] = models_results['F_C1_C3_model'] + models_results['F_C5+_model']

In [ ]:
models_results_parity = pd.DataFrame()
models_results_parity['F_model'] = models_results[['F_H2_s_model', 'F_CO_s_model',
                                       'F_C1_C3_model', 'F_C5+_model', 'F_H2O_s_model', 'F_HCs_model',
                                       ]].melt()['value']
models_results_parity['F_exp'] = models_results[['F_H2_s_exp', 'F_CO_s_exp',
                                      'F_C1_C3_exp', 'F_C5+_exp', 'F_H2O_s_exp', 'F_HCs_exp',
                                       ]].melt()['value']
models_results_parity['component'] = models_results[['F_H2_s_model', 'F_CO_s_model',
                                       'F_C1_C3_model', 'F_C5+_model', 'F_H2O_s_model', 'F_HCs_model',
                                       ]].melt()['variable']
models_results_parity['model'] = models_results[['F_H2_s_model', 'F_CO_s_model',
                                       'F_C1_C3_model', 'F_C5+_model', 'F_H2O_s_model', 'F_HCs_model', 'model'
                                       ]].melt(id_vars = 'model')['model']
models_results_parity = models_results_parity.replace('F_H2_s_model', '$H_2$')
models_results_parity = models_results_parity.replace('F_CO_s_model', '$CO$')
models_results_parity = models_results_parity.replace('F_C1_C3_model', '$C_1 - C_3$')
models_results_parity = models_results_parity.replace('F_C5+_model', '$C_{5+}$')
models_results_parity = models_results_parity.replace('F_HCs_model', '$C_n H_x$')
models_results_parity = models_results_parity.replace('F_H2O_s_model', '$H_2O$')

# models_results_parity['model'] = models_results[['model']].melt()['value']
models_results_parity

In [ ]:
fig_grid_size  = len(minimums)
fig_grid_size

In [ ]:
models_results_parity

In [ ]:
colunas = round(fig_grid_size**0.5)
linhas = round(fig_grid_size**0.5)
fig, ax = plt.subplots(linhas,colunas,figsize=(8,6.5))
plt.subplots_adjust(hspace=0.4, wspace = 0.4)

linha = 0
coluna = 0
for n in range(fig_grid_size):
    sns.scatterplot(
                    data=models_results_parity.loc[
                        ((models_results_parity['model'] == minimums['model'][n])
                                                   & ((models_results_parity['component'] 
                                                   == '$H_2$')
                                                   | (models_results_parity['component'] 
                                                   == '$CO$')
                                                   | (models_results_parity['component'] 
                                                   == '$H_2O$')
                                                   | (models_results_parity['component'] 
                                                   == '$C_n H_x$')
                                                     )
                        )], 
                    x="F_exp", 
                    y="F_model", 
                    hue = 'component',
                    style = 'component',
                    palette = 'bright',
                    ax = ax[linha, coluna],
#                     legend = False,
    )
    ax[linha, coluna].set_title('Model' + str(n+1))
    ax[linha, coluna].grid('both')
    ax[linha, coluna].set_xlabel(None)
    ax[linha, coluna].set_ylabel(None)
    ax[2, coluna].set_xlabel('$F_{j,exp}$ (mol/s)')
    ax[linha, 0].set_ylabel('$F_{j,model}$ (mol/s)')
    ax[linha, coluna].plot(np.linspace(models_results_parity['F_exp'].min()*0.5,
                 models_results_parity['F_exp'].max()*1.2,
                 num = 10), 
     np.linspace(models_results_parity['F_exp'].min()*0.5,
                 models_results_parity['F_exp'].max()*1.2,
                 num = 10))
    ax[linha, coluna].ticklabel_format(axis='both', style='sci', scilimits=(0,0))
    ax[linha, coluna].set_yscale('log')
    ax[linha, coluna].set_xscale('log')
    ax[linha, coluna].get_legend().set_visible(False)
    coluna = coluna + 1
    if (coluna>=colunas): 
        coluna = 0
        linha = linha + 1

#         ax[linha, coluna].set_yscale('log')
# ax[0,0].set_title(r'$-r^,_{FTS} = k^, \frac{P_{{H2}}^a P_{CO}^b}{(1 + K_{CO} P_{CO}^c)^2}$')

ax[2, 0].get_legend().set_visible(True)
ax[2, 0].legend(bbox_to_anchor=(2.8, -0.4), ncols = 4)
fig_name = 'test_parity_F_out_all.pdf'
plt.savefig('/home/rafael/GoogleDrive/uem/Doutorado/Papers/SAF_kinetics/figures/'+fig_name, bbox_inches = 'tight')
plt.show()

In [ ]:
models_results_parity['Residuals (mol/s)'] = models_results_parity['F_exp'] - models_results_parity['F_model']

In [ ]:
colunas = round(fig_grid_size**0.5)
linhas = round(fig_grid_size**0.5)
fig, ax = plt.subplots(linhas,colunas,figsize=(8,6.5))
plt.subplots_adjust(hspace=0.5, wspace = 0.4)

linha = 0
coluna = 0
for n in range(fig_grid_size):
    sns.histplot(
                    data=models_results_parity.loc[
                        ((models_results_parity['model'] == minimums['model'][n])
                                                   & ((models_results_parity['component'] 
                                                   == '$H_2$')
                                                   | (models_results_parity['component'] 
                                                   == '$CO$')
                                                   | (models_results_parity['component'] 
                                                   == '$H_2O$')
                                                   | (models_results_parity['component'] 
                                                   == '$C_n H_x$')
                                                     )
                        )], 
                    x="Residuals (mol/s)", 
                    ax = ax[linha, coluna],
                    kde = True,
#                     bins = 20,
#                     legend = False,
    )
    ax[linha, coluna].set_title('Model' + str(n+1))
    ax[linha, coluna].grid('both')
    ax[linha, coluna].set_xlabel(None)
    ax[linha, coluna].set_ylabel(None)
    ax[2, coluna].set_xlabel('Res. (mol/s)')
    ax[linha, 0].set_ylabel('Count')
#     ax[linha, coluna].ticklabel_format(axis='both', style='sci', scilimits=(0,0))
#     ax[linha, coluna].get_legend().set_visible(False)
    coluna = coluna + 1
    if (coluna>=colunas): 
        coluna = 0
        linha = linha + 1

#         ax[linha, coluna].set_yscale('log')
# ax[0,0].set_title(r'$-r^,_{FTS} = k^, \frac{P_{{H2}}^a P_{CO}^b}{(1 + K_{CO} P_{CO}^c)^2}$')
# ax[2, 0].get_legend().set_visible(True)
# ax[2, 0].legend(bbox_to_anchor=(2.8, -0.3), ncols = 5)
fig_name = 'residuals_histplot.pdf'
plt.savefig('/home/rafael/GoogleDrive/uem/Doutorado/Papers/SAF_kinetics/figures/'+fig_name, bbox_inches = 'tight')
plt.show()